In [16]:
# Re-import necessary libraries due to environment reset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasketch import MinHash, MinHashLSH

# Reload data
student_df = pd.read_csv("StudentAddresses-2016-2024.csv")
rental_df = pd.read_csv("current_rental_reg.csv")

/var/folders/nb/ny5zmrvj7kngqb14cq5n44f00000gn/T/ipykernel_52912/2938342909.py:8: DtypeWarning: Columns (4,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  student_df = pd.read_csv("StudentAddresses-2016-2024.csv")


In [34]:
# Fix ZIP codes
student_df['6e. zip'] = student_df['6e. zip'].astype(str).apply(lambda z: (z if z.startswith('0') else '0' + z)[:5])

# Define suffix abbreviation dictionary
suffix_abbrev =  {
    "STREET": "ST", "ST": "ST", "ST.": "ST", "STR": "ST", "STEET": "ST", "STREET,": "ST", "STREET.": "ST",
    "AVENUE": "AVE", "AVE": "AVE", "AVE.": "AVE", "AV": "AVE",
    "BOULEVARD": "BLVD", "BLVD": "BLVD", "BLVD.": "BLVD",
    "ROAD": "RD", "RD": "RD", "RD.": "RD", "ROA": "RD",
    "DRIVE": "DR", "DR": "DR", "DR.": "DR", "DRIV": "DR",
    "COURT": "CT", "CT": "CT", "CT.": "CT", "COURT," : "CT",
    "LANE": "LN", "LN": "LN", "LN.": "LN",
    "PLACE": "PL", "PL": "PL", "PL.": "PL", "PLAZA": "PL",
    "TERRACE": "TER", "TER": "TER", "TER.": "TER",
    "CIRCLE": "CIR", "CIR": "CIR", "CIRCUIT": "CIR",
    "PARKWAY": "PKWY", "PKWY": "PKWY", "PK": "PKWY",
    "SQUARE": "SQ", "SQ": "SQ", "SQ.": "SQ",
    "WAY": "WAY", "WAY.": "WAY", "WY": "WAY",
    "HIGHWAY": "HWY", "HWY": "HWY",
    "ALLEY": "ALY", "ALY": "ALY"
}

# Student address standardization
def standardize_student_address_no_unit(row):
    street_number = str(row['6a. street #']).strip()
    street_name = str(row['6b. street name']).strip()
    suffix = str(row['6c. street suffix']).strip().upper()
    suffix = suffix_abbrev.get(suffix, suffix)
    zip_code = str(row['6e. zip']).strip()
    address = f"{street_number} {street_name} {suffix}, MA {zip_code}"
    return address.upper().replace("  ", " ").strip()

student_df["standardized_address"] = student_df.apply(standardize_student_address_no_unit, axis=1)

# Rental address cleaning
def clean_registered_address(addr):
    if pd.isna(addr):
        return np.nan
    addr = addr.upper().strip().replace("  ", " ")
    zip_code = addr[-5:] if addr[-5:].isdigit() else ""
    addr_main = addr.split(",")[0]
    tokens = addr_main.split()
    if tokens and len(tokens[-1]) <= 5 and any(c.isdigit() for c in tokens[-1]) and not tokens[-1][-1].isalpha():
        tokens = tokens[:-1]
    if len(tokens) >= 2 and tokens[-1] in suffix_abbrev:
        tokens[-1] = suffix_abbrev[tokens[-1]]
    base = " ".join(tokens)
    return f"{base}, MA {zip_code}"

rental_df["standardized_address"] = rental_df["RegisteredAddress"].apply(clean_registered_address)


In [49]:
import re
from datasketch import MinHash, MinHashLSH

def extract_street_number(address):
    """Extracts the leading number from the address (ignores ranges like 22-24)."""
    match = re.match(r'^(\d+)', address)
    return int(match.group(1)) if match else None

def query_lsh_with_number_check(student_address):
    # Build MinHash for student address
    mh = MinHash(num_perm=128)
    for token in student_address.lower().split():
        mh.update(token.encode('utf8'))

    # Query similar rental addresses
    results = rental_lsh.query(mh)
    student_num = extract_street_number(student_address)

    for match in results:
        rental_num = extract_street_number(match)
        if student_num == rental_num:
            return match  # Only accept match with same number

    return None  # No good match

student_df_unique['matched_address'] = student_df_unique['standardized_address'].apply(query_lsh_with_number_check)
student_df_unique['registration_status'] = student_df_unique['matched_address'].apply(
    lambda x: 'Registered (Fuzzy)' if pd.notna(x) else 'Unregistered'
)



In [50]:
student_df_unique.head(50)

,6a. street #,6b. street name,6c. street suffix,6d. unit #,6e. zip,7. undergraduate (u) or graduate (g),8. full-time (ft) or part-time (pt),9. at-home or not-at-home,9. 5 or more undergrads/unit (y/n),university,year,standardized_address,matched_address,registration_status
0,10,Higgins,ST,NaN,02134,U,FT,NaN,NaN,Emmanuel College,2018-2019,"10 HIGGINS ST, MA 02134",None,Unregistered
2,1189,Commonwealth,AVE,6,02134,U,FT,NaN,NaN,Emmanuel College,2018-2019,"1189 COMMONWEALTH AVE, MA 02134","1189 COMMONWEALTH AVE, MA 02134",Registered (Fuzzy)
3,12,Glenville,AVE,NaN,02134,U,FT,NaN,NaN,Emmanuel College,2018-2019,"12 GLENVILLE AVE, MA 02134",None,Unregistered
5,12,Saunders,ST,NaN,02134,U,FT,NaN,NaN,Emmanuel College,2018-2019,"12 SAUNDERS ST, MA 02134",None,Unregistered
6,1251,Commonwealth,AVE,3,02134,U,FT,NaN,NaN,Emmanuel College,2018-2019,"1251 COMMONWEALTH AVE, MA 02134","1251 COMMONWEALTH AVE, MA 02134",Registered (Fuzzy)
7,17,Highgate,ST,NaN,02134,U,FT,NaN,NaN,Emmanuel College,2018-2019,"17 HIGHGATE ST, MA 02134","17 HIGHGATE ST, MA 02134",Registered (Fuzzy)
8,28,Linden,ST,NaN,02134,U,FT,NaN,NaN,Emmanuel College,2018-2019,"28 LINDEN ST, MA 02134",None,Unregistered
9,28,Quint,AVE,48,02134,U,FT,NaN,NaN,Emmanuel College,2018-2019,"28 QUINT AVE, MA 02134","28 QUINT AVE, MA 02134",Registered (Fuzzy)
10,40,Brainerd,RD,NaN,02134,U,FT,NaN,NaN,Emmanuel College,2018-2019,"40 BRAINERD RD, MA 02134",None,Unregistered
11,49,Pratt,ST,NaN,02134,U,FT,NaN,NaN,Emmanuel College,2018-2019,"49 PRATT ST, MA 02134",None,Unregistered


In [51]:
status_counts = student_df_unique['registration_status'].value_counts()
total = status_counts.sum()

summary_df = pd.DataFrame({
    'Count': status_counts,
    'Percentage': round((status_counts / total) * 100, 2)
})
print(summary_df)


                     Count  Percentage
registration_status                   
Unregistered         72156       87.39
Registered (Fuzzy)   10412       12.61


In [ ]:
# Filter for recent 4 years
recent_years = ['2020-2021', '2021-2022', '2022-2023', '2023-2024']
merged_recent = merged_df[merged_df['year'].isin(recent_years)]

# Group and calculate percentages
status_counts_recent = merged_recent.groupby(['university', 'registration_status']).size().unstack(fill_value=0)
status_counts_recent['Total'] = status_counts_recent.sum(axis=1)
status_counts_recent['% Registered'] = round((status_counts_recent.get('Registered', 0) / status_counts_recent['Total']) * 100, 2)
status_counts_recent['% Unregistered'] = round((status_counts_recent.get('Unregistered', 0) / status_counts_recent['Total']) * 100, 2)

# Plot top 10
top_schools_recent = status_counts_recent.sort_values("Total", ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 6))
top_schools_recent[['% Registered', '% Unregistered']].plot(kind='bar', stacked=True, ax=ax, colormap='coolwarm')

plt.ylabel("Percentage of Addresses")
plt.title("Rental Registration Status by University (2020–2024)")
plt.xticks(rotation=45, ha='right')
plt.legend(loc='upper right')
plt.tight_layout()
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()